## Communication Topology and Belief Dynamics in Multi-Agent LLM Reasoning

### Notebook: [nbviewer link](https://nbviewer.org/github/SafoanMiah/llm-multi-agent-reasoning/blob/main/analysis/analysis.ipynb)   |   Project: [github link](https://github.com/SafoanMiah/llm-multi-agent-reasoning/tree/main)

This notebook analyses experimental results from a multi-agent LLM system tested across different communication topologies on the GSM8K benchmark.

**Topologies tested:**
- **Independent**: agents reason alone, answers aggregated via majority vote
- **Fully Connected**: all agents see each other's responses before revising
- **Mediator**: a mediator summarises responses; agents see only the summary
- **Chain**: agents answer sequentially, each seeing only the previous agent

**Primary questions:**
1. Does collaboration improve accuracy over independent reasoning?
2. How do different communication structures affect convergence?
3. What are the cost/accuracy trade-offs across topologies?

In [22]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

import pandas as pd
import warnings
from pathlib import Path

from mlxtend.evaluate import cochrans_q, mcnemar, mcnemar_table
from itertools import combinations
import numpy as np

---
### 1. Setup & Data Loading

In [23]:
pio.templates.default = "plotly_white"
pio.renderers.default = "notebook_connected+vscode"

warnings.filterwarnings("ignore")

TOPO_COLORS = {
    "independent": "#4A6FA5",
    "full": "#B5534A",
    "mediator": "#4A8F6E",
    "chain": "#7B6BA0",
    "self-refine": "#E8A87C"
}

AGENT_COLORS = {
    "gemma3:4b": "#C06050",
    "phi4-mini": "#5A9E94",
    "llama3.2:3b": "#5A8A6A",
    "qwen2.5:3b-instruct": "#C4A94D",
}

AGENT_NAME_MAP = {
    0: "gemma3:4b", 
    1: "phi4-mini", 
    2: "llama3.2:3b", 
    3: "qwen2.5:3b-instruct"
}

In [24]:
# 4 Agents, 5 Rounds, 100 Questions, Seed 0
folder = "R5-Q200-S0"
result_files = list(Path(f"../results/{folder}").glob("*.csv"))
print(f"Found {len(result_files)} result files:")
for f in result_files:
    print(f"  {f}")
df = pd.concat((pd.read_csv(f) for f in result_files), ignore_index=True)

Found 6 result files:
  ..\results\R5-Q200-S0\chain_20260314_045634.csv
  ..\results\R5-Q200-S0\full_20260315_194735.csv
  ..\results\R5-Q200-S0\independent_20260313_213131.csv
  ..\results\R5-Q200-S0\mediator_20260315_150046.csv
  ..\results\R5-Q200-S0\self_refine_20260322_232713_phi4m.csv
  ..\results\R5-Q200-S0\self_refine_20260323_002938.csv


In [25]:
print("===== Dataset Summary =====")
print(f"Shape: {df.shape}")
print("\nTopology counts:")
print(df['topology'].value_counts())
print("\nRounds per topology:")
print(df.groupby('topology')['round'].max())
print("\nTemperature:        0.4 \nSamples questions:  200")
print(f"\nParse failure rate: {df['parse_failed'].mean():.2%}")

parse_by_model = df.groupby(["topology", "model"])["parse_failed"].mean().unstack()
print(parse_by_model.map(lambda x: f"{x:.1%}"))

===== Dataset Summary =====
Shape: (13165, 15)

Topology counts:
topology
full           4000
mediator       4000
chain          3200
self_refine    1165
independent     800
Name: count, dtype: int64

Rounds per topology:
topology
chain          4
full           5
independent    1
mediator       5
self_refine    5
Name: round, dtype: int64

Temperature:        0.4 
Samples questions:  200

Parse failure rate: 9.32%
model       gemma3:4b llama3.2:3b phi4-mini phi4:14b qwen2.5:3b-instruct
topology                                                                
chain            2.9%       26.4%      1.2%     nan%                0.2%
full             2.6%       26.0%      0.5%     nan%                0.1%
independent      5.5%       42.0%      0.5%     nan%                1.0%
mediator         2.4%       31.0%      0.3%     nan%                0.1%
self_refine      nan%        nan%     24.5%     4.8%                nan%


---
### 2. Accuracy Comparison Across Topologies

The fundamental question: **does collaboration help?**
We compare group-level accuracy (majority vote) across all topologies.

In [26]:
# Question-level accuracy table (one row per question per topology)
q_level = (
    df.groupby(["topology", "question_idx"])
    .agg(
        correct=("correct", "first"),
        expected=("expected_answer", "first"),
        group_answer=("group_answer", "first"),
        last_round_idx=("round", "idxmax"),
    )
    .reset_index()
)

last_round = df.loc[q_level["last_round_idx"]]

#### Individual Agent Accuracy vs Group Accuracy

Does (majority) voting actually help? Comparing individual agent accuracy to the group's majority-vote accuracy.

In [27]:
# All rows with max round per group (one per agent)
last_round = df.loc[df["round"] == df.groupby(["topology", "question_idx"])["round"].transform("max")]

# Individual agent accuracy
individual_acc = (
    last_round.assign(correct=lambda d: d["answer"] == d["expected_answer"])
    .groupby(["topology", "agent_id"])["correct"].mean()
    .reset_index(name="accuracy")
)
individual_acc["agent_id"] = individual_acc["agent_id"].map(AGENT_NAME_MAP)

# Group accuracy
group_acc = (
    q_level.groupby("topology")["correct"].mean()
    .reset_index(name="accuracy")
    .assign(agent_id="Group Vote")
)

# Individual and group accuracy
combined = pd.concat([individual_acc, group_acc], ignore_index=True)
agent_color_map = {**AGENT_COLORS, "Group Vote": "#000000"}


fig = px.bar(
    combined, x="topology", y="accuracy",
    color="agent_id", barmode="group",
    title="Individual Agent vs Group Accuracy by Topology",
    labels={"accuracy": "Accuracy", "topology": "Topology", "agent_id": ""},
    text=combined["accuracy"].map("{:.1%}".format),
    category_orders={"topology": ["independent", "full", "mediator", "chain"]},
    color_discrete_map=agent_color_map
)

fig.update_traces(textposition="outside")
fig.update_layout(yaxis_range=[0, 1.1])

fig.show()
fig.write_image("figures/individual_vs_group_accuracy.png", scale=3, width=1500)

Collaboration dramatically boosts all models, every agent jumps from 7-24% (independent) to 45-84% under collaborative topologies. However, majority voting only outperforms the best individual agent in mediator (84% vs 83%). In fully connected, chain, and independent, at least one or two agents individually beat the group vote, suggesting that open discussion and sequential passing can dilute strong individual reasoning. Mediator's structured summary appears to be the only topology where aggregation consistently adds value beyond the best single agent.

---
### 3. Confidence Analysis

Are agents overconfident? Does confidence actually predict correctness?

In [28]:
calibration = (
    last_round.groupby("topology")
    .apply(lambda g: pd.Series({
        "Mean Confidence": g["confidence"].mean(),
        "Actual Accuracy (%)": (g["answer"] == g["expected_answer"]).mean() * 100,
    }))
    .reset_index()
)
calibration_melted = calibration.melt(
    id_vars="topology", var_name="Metric", value_name="value"
)

In [29]:
# A well-calibrated agent would have these roughly equal

fig = px.bar(
    calibration_melted,
    y="topology", x="value",
    color="Metric", barmode="group",
    text=calibration_melted["value"].apply(lambda x: f"{x:.1f}%"),
    title="Calibration Gap: Mean Confidence vs Actual Accuracy",
    color_discrete_map={"Actual Accuracy (%)": "#97c7ab", "Mean Confidence": "#D18A84"},
    labels={"value": "Percentage", "topology": "Topology"},
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_range=[0,110])

fig.show()
fig.write_image("figures/calibration_gap.png", width=1200, scale=3)

Agents claimed 90%+ confidence across all topologies regardless of correctness. 
* Confidence is **not a reliable signal** in these small models.

---
### 4. Convergence Dynamics
For multi-round topologies (fully connected, mediator), do agents converge
toward agreement? And does that agreement move toward the **correct** answer?

In [30]:

multi_round = df[df["topology"].isin(["full", "mediator"])]

# Agreement per round
avg_agreement = (
    multi_round.groupby(["topology", "question_idx", "round"])["answer"]
    .apply(lambda a: (a.dropna() == a.dropna().mode().iloc[0]).mean() if len(a.dropna()) else 0)
    .reset_index(name="agreement")
    .groupby(["topology", "round"], as_index=False)["agreement"].mean()
)

# Group accuracy per round
avg_acc_round = (
    multi_round.groupby(["topology", "question_idx", "round"])
    .apply(lambda g: int(g["answer"].dropna().mode().iloc[0] == g["expected_answer"].iloc[0]) if len(g["answer"].dropna()) else 0)
    .reset_index(name="correct")
    .groupby(["topology", "round"], as_index=False)["correct"].mean()
)

conf_round = (multi_round.groupby(["topology", "round"])["confidence"].mean().reset_index())

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Agent Agreement Rate Across Rounds", "Group Accuracy Across Rounds", "Mean Confidence Across Rounds")
)

for topo in avg_agreement["topology"].unique():
    fig.add_trace(
        go.Scatter(
            x=avg_agreement[avg_agreement["topology"] == topo]["round"],
            y=avg_agreement[avg_agreement["topology"] == topo]["agreement"],
            name=topo, line=dict(color=TOPO_COLORS.get(topo))
        ),
        row=1, col=1
    )

for topo in avg_acc_round["topology"].unique():
    fig.add_trace(
        go.Scatter(
            x=avg_acc_round[avg_acc_round["topology"] == topo]["round"],
            y=avg_acc_round[avg_acc_round["topology"] == topo]["correct"],
            name=topo, line=dict(color=TOPO_COLORS.get(topo)),
            showlegend=False
        ),
        row=1, col=2
    )

for topo in conf_round["topology"].unique():
    d = conf_round[conf_round["topology"] == topo]
    fig.add_trace(
        go.Scatter(
            x=d["round"], y=d["confidence"],
            name=topo, line=dict(color=TOPO_COLORS[topo]),
            showlegend=False
        ),
        row=1, col=3
    )

fig.update_layout(
    yaxis_tickformat=".0%", yaxis_range=[0, 1.05],
    yaxis2_tickformat=".0%", yaxis2_range=[0, 1.05],
    yaxis3_range=[0, 100]
)

fig.show()
fig.write_image("figures/agreement_accuracy_across_rounds.png", width=1700, scale=3)

Both topologies show strong convergence. Agreement rises from ~45% in round 1 to ~85% by round 5, with accuracy following the same upward trend (21% -> 80%). Confirming agents are converging toward correct answers, not just blindly agreeing. Notably, mediator reaches higher accuracy faster in round 2 and 3, suggesting the mediator's structured summary helps agents focus on the right answer more efficiently. Meanwhile, mean confidence barely changes (~93% throughout), reinforcing that confidence, in the way I've implemented it, is not a meaningful signal in these models.

---
### 5. Answer Flow Between Rounds (Sankey)

How do agents change between rounds? Tracks transitions: correct->correct, correct->incorrect, incorrect->correct, incorrect->incorrect.

In [31]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "sankey"}, {"type": "sankey"}]],
    subplot_titles=["Full", "Mediator"],
)

for i, topo in enumerate(["full", "mediator"]):
    topo_df = multi_round[multi_round["topology"] == topo]
    
    transitions = {"C->C": 0, "C->I": 0, "I->C": 0, "I->I": 0}
    for r in range(1, topo_df["round"].max()):
        curr = topo_df[topo_df["round"] == r].set_index(["question_idx", "agent_id"])
        nxt = topo_df[topo_df["round"] == r + 1].set_index(["question_idx", "agent_id"])
        joined = curr[["answer", "expected_answer"]].join(nxt[["answer"]], rsuffix="_next", how="inner")
        
        c = joined["answer"] == joined["expected_answer"]
        n = joined["answer_next"] == joined["expected_answer"]
        transitions["C->C"] += (c & n).sum()
        transitions["C->I"] += (c & ~n).sum()
        transitions["I->C"] += (~c & n).sum()
        transitions["I->I"] += (~c & ~n).sum()

    values = [transitions["C->C"], transitions["C->I"], transitions["I->C"], transitions["I->I"]]
    total = sum(values)
    pct = [v / total * 100 for v in values]
    
    fig.add_trace(go.Sankey(
        node=dict(
            pad=20, thickness=25,
            label=[
                f"Correct (R1)\n{(transitions['C->C'] + transitions['C->I']) / total:.0%}",
                f"Incorrect (R1)\n{(transitions['I->C'] + transitions['I->I']) / total:.0%}",
                f"Correct (R5)\n{(transitions['C->C'] + transitions['I->C']) / total:.0%}",
                f"Incorrect (R5)\n{(transitions['C->I'] + transitions['I->I']) / total:.0%}",
            ],
            color=["#51c280", "#cf786e", "#51c280", "#cf786e"],
        ),
        link=dict(
            source=[0, 0, 1, 1], target=[2, 3, 2, 3],
            value=values,
            label=[f"{v:,}" for v in values],
            color=["rgba(46,204,113,0.4)", "rgba(231,76,60,0.4)",
                   "rgba(46,204,113,0.4)", "rgba(231,76,60,0.4)"],
        ),
    ), row=1, col=i + 1)

fig.update_layout(title="Answer Transitions Between Rounds")

fig.show()
fig.write_image("figures/sankey_transitions.png", width=1700, scale=3)

Both topologies show a net positive correction effect. Starting from ~36-39% individual correctness in round 1, agents improve to ~61-64% by round 5, a ~25 percentage point gain through discussion alone. The dominant flow is Incorrect->Correct, confirming that collaboration helps agents fix mistakes. A small Correct->Incorrect flow exists in both (agents occasionally getting talked out of right answers), but it is far outweighed by the corrections. The two topologies perform similarly here, though mediator shows a slightly thinner C->I band, suggesting the structured summary helps correct agents from being swayed. 
* Note: that final group accuracy exceeds the individual shown here because majority voting further filters out remaining incorrect minority answers.

---
### 6. Per-Question Error Analysis

Which questions were hardest? Did certain topologies rescue questions that others got wrong?

**How to read:** 
* Diagonal = total questions solved. 
* Off-diagonal = questions solved by row but NOT column.

In [32]:

q_pivot = q_level.pivot_table(index="question_idx", columns="topology", values="correct", aggfunc="first").fillna(0)

topos = [t for t in ["independent", "full", "mediator", "chain"] if t in q_pivot.columns]
solved_by = {t: set(q_pivot[q_pivot[t] == 1].index) for t in topos}

overlap_data = []
for t1 in topos:
    for t2 in topos:
        if t1 == t2:
            overlap_data.append({"from": t1, "to": t2, "value": len(solved_by[t1])})
        else:
            overlap_data.append({"from": t1, "to": t2, "value": len(solved_by[t1] - solved_by[t2])})

overlap_df = pd.DataFrame(overlap_data).pivot(index="from", columns="to", values="value")
overlap_df = overlap_df.loc[topos, topos]

fig = px.imshow(
    overlap_df,
    text_auto=True,
    color_continuous_scale="RdYlGn",
    title="Question Overlap Between Topologies",
    labels=dict(x="Topology", y="Topology", color="Questions"),
)

fig.show()
fig.write_image("figures/question_overlap.png", scale=3)

Mediator is the dominant topology. It solves the most questions and is a complete superset of independent, every question independent solved, mediator also solved (off-diagonal = 0). Full comes second but only a few of those are unique to it; mediator captures almost everything full does. Chain solves a good amount but only a couple are unique compared to mediator. All suggesting that structured mediation provides the broadest coverage, and combining topologies would offer minimal additional gain over mediator alone.

---
### 7. Token Cost & Efficiency
Collaboration costs tokens. Is it worth it?

In [33]:
token_usage = (
    df.groupby(["topology", "question_idx"])
    .agg(
        total_prompt=("prompt_tokens", "sum"),
        total_completion=("completion_tokens", "sum"),
        correct=("correct", "first"),
    )
    .assign(total_tokens=lambda d: d["total_prompt"] + d["total_completion"])
    .reset_index()
)

In [34]:
# Per round accuracy and cumulative tokens for iterative topologies
scatter_rows = []
for topo in ["full", "mediator"]:
    topo_df = df[df["topology"] == topo]
    for r in range(1, topo_df["round"].max() + 1):
        up_to_r = topo_df[topo_df["round"] <= r]
        round_r = topo_df[topo_df["round"] == r]
        tokens = up_to_r.groupby("question_idx")[["prompt_tokens", "completion_tokens"]].sum().sum(axis=1).mean()
        acc = (
            round_r.groupby("question_idx")
            .apply(lambda g: int(g["answer"].dropna().mode().iloc[0] == g["expected_answer"].iloc[0]) if len(g["answer"].dropna()) else 0)
            .mean()
        )
        scatter_rows.append({"topology": topo, "round": r, "label": f"{topo} R{r}", "accuracy": acc, "tokens": tokens})

# Single points for non iterative topologies
for topo in ["independent", "chain"]:
    topo_t = token_usage[token_usage["topology"] == topo]
    scatter_rows.append({"topology": topo, "round": 1, "label": topo, "accuracy": topo_t["correct"].mean(), "tokens": topo_t["total_tokens"].mean()})

scatter_df = pd.DataFrame(scatter_rows)

In [35]:

fig = px.scatter(
    scatter_df, x="tokens", y="accuracy",
    color="topology", color_discrete_map=TOPO_COLORS, text="label",
    title="Accuracy vs Token Cost (Per Round for Iterative Topologies)",
    labels={"tokens": "Mean Tokens per Question", "accuracy": "Accuracy"},
)
fig.update_traces(textposition="top center")
fig.update_layout(yaxis_tickformat=".0%", showlegend=False)

fig.show()
fig.write_image("figures/accuracy_vs_cost.png", width=1200, scale=3)

---
### 8. Summary Table

In [36]:
rows = []
for topo in ["independent", "chain"]:
    row = scatter_df[scatter_df["topology"] == topo].iloc[0]
    accuracy = row['accuracy']
    tokens = row['tokens']
    rows.append({
        "Topology": topo.title(),
        "Accuracy": f"{accuracy:.1%}",
        "Tokens/Q": f"{tokens:,.0f}",
        "Tokens/Correct": f"{tokens / accuracy:,.0f}",
        "Parse Fail %": f"{df[df['topology'] == topo]['parse_failed'].mean():.1%}"
    })

for topo in ["mediator", "full"]:
    topo_rows = scatter_df[scatter_df["topology"] == topo]
    pf = f"{df[df['topology'] == topo]['parse_failed'].mean():.1%}"
    for r in [1, 2, 3, 4, topo_rows["round"].max()]:
        row = topo_rows[topo_rows["round"] == r].iloc[0]
        accuracy = row['accuracy']
        tokens = row['tokens']
        rows.append({
            "Topology": f"{topo.title()}: round {r}",
            "Accuracy": f"{accuracy:.1%}",
            "Tokens/Q": f"{tokens:,.0f}",
            "Tokens/Correct": f"{tokens / accuracy:,.0f}",
            "Parse Fail %": pf
        })

summary = pd.DataFrame(rows)

In [37]:
fig = go.Figure(go.Table(
    header=dict(values=list(summary.columns), fill_color="#3D4854",
                font=dict(color="white", size=14), align="center"),
    cells=dict(values=[summary[col] for col in summary.columns],
               font=dict(size=13), align="center", height=30),
))

fig.update_layout(title="Experiment Summary", height=500, margin=dict(b=5))

fig.show()
fig.write_image("figures/experiment_summary_table.png", width=1000, scale=3)

Mediator R2 is a sweet spot, 75.5% accuracy at just 14k tokens/Q and onw of the lowest cost per correct answer (~19k). Adding more rounds increases the token cost a lot with diminishing returns: R3 gains 5% accuracy for 5k more tokens. Fully connected follows the same pattern but consistently lags behind mediator in accuracy even with the lowest average Tokens/Correct. Independent is deceptively cheap (6k tokens) but its 32k tokens/correct is higher than mediator R2 because it only gets 19% right, chain is terrible value, highest tokens/correct (34k) at mediocre accuracy (63%).

---
### 9. Statistical Validation

Are the accuracy differences between topologies statistically significant, or could they be noise from a 100-question sample?

We apply a standard three-step testing pipeline:
1. **Cochran's Q test**: omnibus test, is there any difference at all? (Raschka, 2018; Fleiss et al., 2003)
2. **Pairwise McNemar's tests**: post-hoc, where exactly are the differences? (Dietterich, 1998)
3. **Bootstrap confidence intervals**: uncertainty quantification, how stable are these accuracy numbers? (Efron & Tibshirani, 1993)

All three are appropriate because our data is **paired** (same 100 questions across all topologies) and **binary** (correct/incorrect per question).

In [39]:
# For iterative topologies, we select the round that minimises **tokens per correct answer**.
TOPO_ORDER = ["independent", "chain", "full", "mediator"]

q_correct = (
    df.groupby(["topology", "question_idx"])["correct"]
    .first()
    .reset_index()
    .pivot(index="question_idx", columns="topology", values="correct")
    .fillna(0)
    .astype(int)[TOPO_ORDER]
)
print(f"Questions: {len(q_correct)}")
print(f"Final-round accuracies: { {t: f'{q_correct[t].mean():.1%}' for t in TOPO_ORDER} }")

Questions: 200
Final-round accuracies: {'independent': '18.0%', 'chain': '63.0%', 'full': '78.0%', 'mediator': '84.0%'}


#### Step 1: Cochran's Q Test (Omnibus)

Tests $H_0$: all four topologies have **equal accuracy**.

In [40]:
Q_stat, p_cochran = cochrans_q(*[q_correct[t].values for t in TOPO_ORDER])
print(f"Cochran's Q = {Q_stat:.2f}, p = {p_cochran:.2e}")
print(f"-> {'Reject H0' if p_cochran < 0.05 else 'Fail to reject H0'} (alpha = 0.05)")

Cochran's Q = 32.12, p = 1.06e-07
-> Reject H0 (alpha = 0.05)


#### Step 2: Pairwise McNemar's Tests (Post-Hoc)

McNemar's examines **discordant pairs**; questions where one topology got it right and the other didn't.
* **Bonferroni correction** ($\alpha = 0.05 / 6 = 0.0083$) controls for multiple comparisons.


In [41]:
# Which is more accurate
pairs = list(combinations(TOPO_ORDER, 2))
n_pairs = len(pairs)
y_target = np.ones(len(q_correct), dtype=int)
mcnemar_results = []

for topo_a, topo_b in pairs:
    ct = mcnemar_table(y_target, q_correct[topo_a].values, q_correct[topo_b].values)
    b, c = ct[0, 1], ct[1, 0]
    chi2, p_raw = mcnemar(ct, exact=(b + c) < 25, corrected=True)
    p_adj = min(p_raw * n_pairs, 1.0)
    sig = "***" if p_adj < 0.001 else "**" if p_adj < 0.01 else "*" if p_adj < 0.05 else "ns"
    mcnemar_results.append({"Pair": f"{topo_a} vs {topo_b}", "A only Correct": b, "B only Correct": c, "p (raw)": f"{p_raw:.4e}", "p (Bonferroni)": f"{p_adj:.4e}", "Sig": sig})

mcnemar_df = pd.DataFrame(mcnemar_results)

In [42]:
fig = go.Figure(go.Table(
    header=dict(values=list(mcnemar_df.columns), fill_color="#3D4854", font=dict(color="white", size=13), align="center"),
    cells=dict(values=[mcnemar_df[col] for col in mcnemar_df.columns], font=dict(size=12), align="center", height=28),
))
fig.update_layout(title="Pairwise McNemar's Tests (Bonferroni corrected)", height=350, margin=dict(b=0))

fig.show()
fig.write_image("figures/mcnemar_table.png", width=1500, scale=3)

All collaborative topologies outperform independent (p < 0.001), confirming that multi-agent communication improves reasoning regardless of structure. Among them, mediator and full both significantly outperform chain (p < 0.001), while full vs mediator is not significant after Bonferroni correction (p = 0.40) which suggests that at final round accuracy, structured mediation and open discussion converge to similar performance.

#### Step 3: Bootstrap Pareto Frontier

Each point on the accuracy vs tokens scatter represents a topology at a specific round. Adding bootstrap 95% confidence intervals to show the uncertainty around each point, then identify which configurations are **Pareto optimal**, not dominated by any other on both accuracy and cost.

In [43]:
np.random.seed(42)
N_BOOT = 10_000

boot_cis = []
for _, row in scatter_df.iterrows():
    topo, r = row["topology"], int(row["round"])
    topo_df = df[df["topology"] == topo]

    round_df = topo_df[topo_df["round"] == r] if topo in ["full", "mediator"] else topo_df
    q_acc = round_df.groupby("question_idx").apply(
        lambda g: int(g["answer"].dropna().mode().iloc[0] == g["expected_answer"].iloc[0]) if len(g["answer"].dropna()) else 0
    )

    up_to = topo_df[topo_df["round"] <= r] if topo in ["full", "mediator"] else topo_df
    q_tok = up_to.groupby("question_idx")[["prompt_tokens", "completion_tokens"]].sum().sum(axis=1)

    common = q_acc.index.intersection(q_tok.index)
    acc_v, tok_v = q_acc.loc[common].values, q_tok.loc[common].values
    idx = np.random.randint(0, len(acc_v), (N_BOOT, len(acc_v)))

    boot_cis.append({
        "label": row["label"], "topology": topo, "acc": acc_v.mean(), "tok": tok_v.mean(),
        "acc_lo": np.percentile(acc_v[idx].mean(axis=1), 2.5),
        "acc_hi": np.percentile(acc_v[idx].mean(axis=1), 97.5),
        "tok_lo": np.percentile(tok_v[idx].mean(axis=1), 2.5),
        "tok_hi": np.percentile(tok_v[idx].mean(axis=1), 97.5),
    })

boot_df = pd.DataFrame(boot_cis)

In [44]:
boot_df["pareto"] = [
    not any(
        (boot_df.loc[j, "acc"] >= row["acc"]) & (boot_df.loc[j, "tok"] <= row["tok"]) &
        ((boot_df.loc[j, "acc"] > row["acc"]) | (boot_df.loc[j, "tok"] < row["tok"]))
        for j in boot_df.index if j != i
    )
    for i, row in boot_df.iterrows()
]

print("Pareto-optimal configurations:")
print(boot_df[boot_df["pareto"]][["label", "acc", "tok"]].to_string(index=False))

# %%
fig = go.Figure()
for _, row in boot_df.iterrows():
    color = TOPO_COLORS[row["topology"]]
    is_pareto = row["pareto"]

    fig.add_trace(go.Scatter(
        x=[row["tok"]], y=[row["acc"]],
        error_x=dict(type="data", symmetric=False, array=[row["tok_hi"] - row["tok"]], arrayminus=[row["tok"] - row["tok_lo"]], color=color, thickness=1.5),
        error_y=dict(type="data", symmetric=False, array=[row["acc_hi"] - row["acc"]], arrayminus=[row["acc"] - row["acc_lo"]], color=color, thickness=1.5),
        mode="markers+text", text=[row["label"]] if is_pareto else [""],
        textposition="top center",
        marker=dict(size=10 if is_pareto else 6, color=color, opacity=1.0 if is_pareto else 0.25,
                    symbol="star" if is_pareto else "circle"),
        showlegend=False,
    ))

pareto_pts = boot_df[boot_df["pareto"]].sort_values("tok")
fig.add_trace(go.Scatter(
    x=pareto_pts["tok"], y=pareto_pts["acc"],
    mode="lines", line=dict(color="grey", dash="dash", width=1.5),
    name="Pareto frontier",
))

fig.update_layout(
    title="Accuracy vs Token Cost with Bootstrap 95% CIs (★ = Pareto-optimal)",
    xaxis_title="Mean Tokens per Question",
    yaxis_title="Accuracy", yaxis_tickformat=".0%",
    height=500,
)
fig.show()
fig.write_image("figures/pareto_frontier_bootstrap.png", width=1200, scale=3)

Pareto-optimal configurations:
      label   acc       tok
    full R1 0.200  5509.345
    full R2 0.625 11461.280
mediator R1 0.215  7736.170
mediator R2 0.755 14081.195
mediator R3 0.805 18790.385
mediator R4 0.825 23135.030


The Pareto frontier is dominated by mediator configurations from R2 onwards, at any given token budget, mediator achieves the highest accuracy. Chain and later full rounds are consistently dominated (below the frontier), meaning a mediator config always exists that is both cheaper and more accurate. The bootstrap CIs confirm these positions are stable: mediator R2's accuracy interval [~70%, 80%] doesn't overlap with full R2's [~58%, 67%], reinforcing the McNemar finding. 

Diminishing returns are visible along the frontier, the jump from R2 to R3 costs ~5k extra tokens for ~5% accuracy, while R3 to R4 costs another ~5k for just ~2%

---
### 10. Baseline 
Comparisons so far we've shown that collaborative topologies outperform independent reasoning. Two questions remain:                                                                                                                                                                                               
1. **Is it the collaboration, or just more thinking time?** Would a single model improve with multiple   rounds of self-refinement
2. **Is it the topology, or just model scale?** Would a single larger model beat collaborative small models

We test two baselines:                                                                                
- **Self-Refine**: Single phi4-mini model, 5 rounds of self-critique (no peer input)
- **Big Model**: Single 14B model in one pass (phi4:14b)

In [ ]:
# Load baseline results
baseline_dfs = {}

# Self-refine
sr_path = f"../results/{folder}/self_refine_20260322_232713.csv"
if Path(sr_path).exists():
    baseline_dfs["self_refine"] = pd.read_csv(sr_path)
    print(f"Loaded self_refine: {baseline_dfs['self_refine'].shape[0]} rows")

Loaded self_refine: 1000 rows


In [ ]:
# Compute baseline metrics
sr = baseline_dfs["self_refine"]
sr_last = sr.loc[sr.groupby('question_idx')['round'].idxmax()]
sr_acc = (sr_last['answer'] == sr_last['expected_answer']).mean()
sr_tok = sr.groupby('question_idx')[['prompt_tokens', 'completion_tokens']].sum().sum(axis=1).mean()

med_df = df[df['topology'] == 'mediator']
med_acc = med_df.groupby('question_idx')['correct'].first().mean()
med_tok = med_df.groupby('question_idx')[['prompt_tokens', 'completion_tokens']].sum().sum(axis=1).mean()

indep_df = df[df['topology'] == 'independent']
indep_acc = indep_df.groupby('question_idx')['correct'].first().mean()
indep_tok = indep_df.groupby('question_idx')[['prompt_tokens', 'completion_tokens']].sum().sum(axis=1).mean()

# Build comparison table
baseline_comp = pd.DataFrame([
    {"Method": "independent", "Accuracy": indep_acc, "Tokens/Q": indep_tok},
    {"Method": "self-refine", "Accuracy": sr_acc, "Tokens/Q": sr_tok},
    {"Method": "mediator", "Accuracy": med_acc, "Tokens/Q": med_tok},
])

print(baseline_comp.to_string(index=False))
print(f"\nSelf-Refine vs Independent: +{sr_acc - indep_acc:.1%}")
print(f"Mediator vs Self-Refine: +{med_acc - sr_acc:.1%}")

     Method  Accuracy  Tokens/Q
independent      0.18   5840.02
self-refine      0.53   3171.82
   mediator      0.84  27899.89

Self-Refine vs Independent: +35.0%
Mediator vs Self-Refine: +31.0%


In [193]:
# Accuracy vs Tokens side by side
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Accuracy", "Tokens per Question"),
    shared_yaxes=True,
)

for _, row in baseline_comp.iterrows():
    name = row["Method"]
    fig.add_trace(go.Bar(
        y=[name], x=[row["Accuracy"]],
        orientation="h",
        text=[f"{row['Accuracy']:.1%}"],
        textposition="inside",
        marker_color=TOPO_COLORS.get(name),
        showlegend=False,
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        y=[name], x=[row["Tokens/Q"]],
        orientation="h",
        text=[f"{row['Tokens/Q']:,.0f}"],
        textposition="inside",
        marker_color=TOPO_COLORS.get(name),
        showlegend=False,
    ), row=1, col=2)

fig.update_layout(
    title="Baseline Comparison: Collaboration vs Single-Model Approaches",
    height=300,
)
fig.show()
fig.write_image("figures/baseline_tradeoff.png", width=1200, scale=3)

#### Interpretation

**Self-Refine (53%) vs Independent (18%)**: Self-refinement alone provides a +35% boost over independent voting, confirming that **multiple rounds of self-critique help a single model improve**. However, it still falls far short of collaborative methods.

**Mediator (84%) vs Self-Refine (53%)**: Adding **multi-agent collaboration** on top of multiple rounds provides an additional +31% boost. This shows that **diversity of perspectives matters more than just thinking time**: a model critiquing itself cannot escape its own blind spots, but seeing other agents reasoning helps it catch errors it would otherwise miss.

**Token tradeoff**: Self-refine uses ~4k tokens while mediator uses ~24k tokens, a 6x increase for a 31% accuracy gain. Whether this is worthwhile depends on whether you value accuracy or efficiency. For applications where correctness is critical (e.g., math tutoring, medical decision support), the collaboration overhead is justified. For lightweight applications, self-refine may be sufficient.